# Création du fichier csv contenant les protéines des fichiers .fasta pour créer les embeddings

In [2]:
import os

# Obtention de la liste de tous les fichiers .fasta
fasta_path = "../PMIND2026_cluster/darkdino_fasta"

# Fichier de sortie
output = "./Data/darkdino_cleaned_for_bert.csv"


## Création du fichier

In [ ]:
def generer_csv():
    """ Génère un fihcier .csv contenant les protéines des fichiers .fasta 
    
    Structure du .csv : id,sequence
        id : id de la protéine dans les fichiers .fasta (ex: Abedinium_dasypus_EP00812|sequence00001)
        sequence : sequence d'acides aminés de la protéine
        
    Les protéines contenant des X ne sont pas comprises dans ce .csv car l'encodeur de ProteinBERT ne les accepte pas.
    """
    # Récupération des fichiers triés des .fasta
    fichiers = sorted([f for f in os.listdir(fasta_path) if f.endswith(('.fasta', '.fa'))])
    
    with open(output, 'w') as f_out:
        # Entête du fichier 
        f_out.write("id,sequence\n")
        
        for fichier in fichiers:
            with open(os.path.join(fasta_path, fichier), 'r') as f_in:
                current_id = ""
                sequence = []
                for line in f_in:
                    line = line.strip()
                    
                    # Ligne avec l'id d'une protéine -> protéine suivante
                    if line.startswith(">"):
                        
                        # Sauvegarde de la protéine précédente
                        if sequence:
                            seq_str = "".join(sequence).upper()
                            
                            # Sauvegarde de la protéine seulement si sa séquence ne contient aucun X
                            if 'X' not in seq_str and len(seq_str) >= 5:
                                f_out.write(f"{current_id},{seq_str}\n")
                                
                        current_id = line[1:].split()[0]
                        sequence = []
                        
                    else:
                        sequence.append(line)
                        
                # Dernière du fichier
                if sequence:
                    seq_str = "".join(sequence).upper()
                    if 'X' not in seq_str and len(seq_str) >= 5:
                        f_out.write(f"{current_id},{seq_str}\n")

generer_csv()

## Statistique du fichier .csv créé et des fichiers .fasta

In [ ]:
# Compter les lignes avec plus de 512 et 1024 acides aminés
total = 0
plus_de_512 = 0
plus_de_1024 = 0

with open(output, 'r') as f:
    for ligne in f:
        ligne = ligne.strip()
        
        if ligne.startswith('id,'):
            continue
        
        len_ligne = len(ligne)
        
        total += 1
        
        if len_ligne > 512:
            plus_de_512 += 1
            
        if len_ligne > 1024:
            plus_de_1024 += 1
            
print(f"Nombre de lignes: {total}")
print(f"Lignes > 512 caractères: {plus_de_512}")
print(f"Lignes > 1024 caractères: {plus_de_1024}")
print(f"Lignes > 512 caractères: {(plus_de_512*100/total):.2f}%")
print(f"Lignes > 1024 caractères: {(plus_de_1024*100/total):.2f}%")

Nombre de lignes: 5717064
Lignes > 512 caractères: 960451
Lignes > 1024 caractères: 153701
Lignes > 512 caractères: 16.80%
Lignes > 1024 caractères: 2.69%


In [ ]:
total_fasta = 0
X_fasta = 0
short_fasta = 0  # Protéines avec < 5 acides aminés (aa)
fichiers = [f for f in os.listdir(fasta_path) if f.endswith(('.fasta', '.fa'))]
    
for nom_fich in fichiers:
    with open(os.path.join(fasta_path, nom_fich), 'r', encoding='utf-8', errors='ignore') as f_in:
        current_id = ""
        sequence = []
        for line in f_in:
            line = line.strip()
            if line.startswith(">"):
                if sequence:
                    seq_str = "".join(sequence).upper()
                    if 'X' in seq_str:
                        X_fasta += 1
                    if len(seq_str) < 5:
                        short_fasta += 1
                    total_fasta += 1
                        
                current_id = line[1:].split()[0]
                sequence = []
                    
            else:
                sequence.append(line)
                    
        # Dernière protéine du fichier
        if sequence:
            seq_str = "".join(sequence).upper()
            if 'X' in seq_str:
                X_fasta += 1
            if len(seq_str) < 5:
                short_fasta += 1
            total_fasta += 1

print(f"Nombre de protéines avec un X dans les .fasta : {X_fasta}")
print(f"Nombre de protéines avec < 5 aa dans les .fasta : {short_fasta}")
print(f"Nombre de protéines des .fasta : {total_fasta}")
print(f"Vérification : {X_fasta} + {short_fasta} + {total_fasta - X_fasta - short_fasta} = {total_fasta}")
print(f"Attendu dans le CSV : {total_fasta - X_fasta - short_fasta}")

Nombre de protéines avec un X dans les .fasta : 1708985
Nombre de protéines avec < 5 aa dans les .fasta : 9
Nombre de protéines des .fasta : 7426058
Vérification : 1708985 + 9 + 5717064 = 7426058
Attendu dans le CSV : 5717064
